Builds gold_skill_demand_monthly (Day 14).
Reads:  jobmarket.silver.silver_posting_skills, silver_job_postings
Writes: jobmarket.gold.gold_skill_demand_monthly
Grain: month x source x skill. Percentages within-source (truncation
asymmetry, Day 12). mom_growth via lag window.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sk = spark.table("jobmarket.silver.silver_posting_skills")
sp = spark.table("jobmarket.silver.silver_job_postings")

# --- numerator: distinct postings per (month, source, skill) ---
skill_monthly = (sk
    .join(sp.select("posting_id", "source"), "posting_id")
    .withColumn("month", F.date_trunc("month", "posted_date").cast("date"))
    .groupBy("month", "source", "skill", "skill_group")
    .agg(F.countDistinct("posting_id").alias("posting_count")))

# --- denominator: ALL postings per (month, source) — not just matched ones ---
totals = (sp
    .withColumn("month", F.date_trunc("month", "posted_date").cast("date"))
    .groupBy("month", "source")
    .agg(F.countDistinct("posting_id").alias("total_postings")))

demand = (skill_monthly
    .join(totals, ["month", "source"])
    .withColumn("pct_of_postings",
        F.round(F.col("posting_count") / F.col("total_postings") * 100, 2)))

# --- month-over-month growth: lag within (source, skill), ordered by month ---
w = Window.partitionBy("source", "skill").orderBy("month")

demand = (demand
    .withColumn("prev_count", F.lag("posting_count").over(w))
    .withColumn("mom_growth_pct",
        F.round((F.col("posting_count") - F.col("prev_count"))
                / F.col("prev_count") * 100, 1))
    .drop("prev_count"))

(demand.write.format("delta").mode("overwrite")
    .saveAsTable("jobmarket.gold.gold_skill_demand_monthly"))

g = spark.table("jobmarket.gold.gold_skill_demand_monthly")
print("Rows:", g.count())
g.orderBy(F.desc("month"), F.desc("posting_count")).limit(10).display()

In [0]:
g.filter("source = 'kaggle_backfill' AND month = '2024-04-01'") \
 .orderBy(F.desc("posting_count")).limit(10).display()